# **Project Name**
- **Regression - Yes Bank Stock Closing Price Prediction**

#### **Project Type**
- Regression

#### **Contribution**
- Individual

#### **Project Theme**
- **Machine Learning & GenAI with Microsoft Azure**


# **Problem Statement**

Yes Bank is one of the well-known private sector banks in India. The bank saw sharp market sentiment changes after the governance concerns and fraud-related news cycle that intensified from 2018 onward. This creates an interesting modeling challenge because stock behavior can shift abruptly during stress periods.

The primary objective of this project is to **predict the monthly closing price of Yes Bank stock** using the monthly `Open`, `High`, `Low`, and engineered time-based features.


# **Business Context**

- Investors and analysts want to understand how market range information relates to the month-end close.
- The project also explores whether a standard regression setup can still remain useful when the bank goes through structural breaks and abnormal sentiment.
- From an interview perspective, the project demonstrates EDA, feature engineering, time-aware evaluation, regularization, explainability, and deployment readiness.


# **Project Checklist**

1. Problem statement and business context
2. Data understanding
3. Dataset loading and cleanup
4. EDA and storytelling
5. Feature engineering and multicollinearity handling
6. Target feature conditioning
7. Train-test split, model fitting, testing, evaluation, and regularization
8. Model explainability
9. Streamlit + Microsoft Azure GenAI deployment plan
10. Conclusion and Git commit checkpoints


# ***Let's Begin !***


## ***1. Know Your Data***


### Import Libraries and Configure the Notebook


In [ ]:
import io
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


### Project Helper Functions


In [ ]:
"""Utilities for the Yes Bank stock closing price regression project."""

import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


SEED = 42
DATE_FORMAT = "%b-%y"

FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "range_value",
    "range_pct",
    "high_open_gap",
    "open_low_gap",
    "close_lag_1",
    "close_lag_3",
    "close_lag_6",
    "close_ma_3",
    "close_ma_6",
    "close_std_3",
    "close_std_6",
    "month_sin",
    "month_cos",
    "year",
    "time_index",
    "post_2018_crisis",
    "covid_shock",
]

TARGET_COLUMN = "Close"


def load_yes_bank_data(csv_path: Union[str, Path]) -> pd.DataFrame:
    """Load and sort the monthly Yes Bank stock price dataset."""
    frame = pd.read_csv(csv_path)
    frame["Date"] = pd.to_datetime(frame["Date"], format=DATE_FORMAT)
    frame = frame.sort_values("Date").reset_index(drop=True)
    return frame


def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create time-aware and market-behavior features used by the models."""
    df = frame.copy()
    df["month"] = df["Date"].dt.month
    df["quarter"] = df["Date"].dt.quarter
    df["year"] = df["Date"].dt.year
    df["time_index"] = np.arange(len(df))

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

    df["range_value"] = df["High"] - df["Low"]
    df["range_pct"] = np.where(df["Open"] != 0, df["range_value"] / df["Open"], 0.0)
    df["high_open_gap"] = df["High"] - df["Open"]
    df["open_low_gap"] = df["Open"] - df["Low"]

    df["post_2018_crisis"] = (df["Date"] >= pd.Timestamp("2018-01-01")).astype(int)
    df["covid_shock"] = (df["Date"] >= pd.Timestamp("2020-03-01")).astype(int)

    for lag in (1, 3, 6):
        df["close_lag_{0}".format(lag)] = df["Close"].shift(lag)

    for window in (3, 6):
        shifted_close = df["Close"].shift(1)
        df["close_ma_{0}".format(window)] = shifted_close.rolling(window=window).mean()
        df["close_std_{0}".format(window)] = shifted_close.rolling(window=window).std()

    return df


def build_modeling_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Return the final modeling frame with complete feature rows only."""
    model_df = engineer_features(frame)
    required_columns = ["Date", TARGET_COLUMN] + FEATURE_COLUMNS
    model_df = model_df[required_columns].dropna().reset_index(drop=True)
    return model_df


def build_preprocessor(feature_columns: Optional[Iterable[str]] = None) -> ColumnTransformer:
    """Build the preprocessing pipeline for numeric features."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[("numeric", numeric_pipeline, columns)],
        remainder="drop",
    )


def make_pipeline(model, feature_columns: Optional[Iterable[str]] = None) -> Pipeline:
    """Create a full preprocessing + model pipeline."""
    return Pipeline(
        steps=[
            ("preprocess", build_preprocessor(feature_columns)),
            ("model", model),
        ]
    )


def compute_metrics(y_true: pd.Series, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute a consistent set of regression metrics."""
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(rmse),
        "R2": float(r2_score(y_true, y_pred)),
    }


def time_aware_split(
    model_df: pd.DataFrame,
    feature_columns: Optional[Iterable[str]] = None,
    test_ratio: float = 0.2,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, pd.DataFrame, pd.DataFrame]:
    """Split the dataset without shuffling to preserve chronology."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    split_index = int(len(model_df) * (1 - test_ratio))

    train_df = model_df.iloc[:split_index].copy()
    test_df = model_df.iloc[split_index:].copy()

    x_train = train_df[columns]
    x_test = test_df[columns]
    y_train = train_df[TARGET_COLUMN]
    y_test = test_df[TARGET_COLUMN]

    return x_train, x_test, y_train, y_test, train_df, test_df


def naive_baseline_metrics(model_df: pd.DataFrame, test_ratio: float = 0.2) -> Dict[str, float]:
    """Use previous month's close as a naive baseline."""
    _, _, _, y_test, _, test_df = time_aware_split(model_df, test_ratio=test_ratio)
    baseline_pred = test_df["close_lag_1"].to_numpy()
    metrics = compute_metrics(y_test, baseline_pred)
    metrics["Model"] = "Naive Previous Close"
    return metrics


def get_candidate_models() -> Dict[str, object]:
    """Return interview-friendly baseline model candidates."""
    return {
        "Lasso Regression": Lasso(alpha=0.2, max_iter=20000),
        "Ridge Regression": Ridge(alpha=1.0),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.9, max_iter=20000),
        "Random Forest": RandomForestRegressor(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=2,
            random_state=SEED,
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            random_state=SEED,
        ),
    }


def evaluate_models(
    model_df: pd.DataFrame,
    feature_columns: Optional[Iterable[str]] = None,
    test_ratio: float = 0.2,
) -> Tuple[pd.DataFrame, Dict[str, Pipeline], Dict[str, object]]:
    """Train and evaluate the candidate models on a chronological holdout set."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    x_train, x_test, y_train, y_test, train_df, test_df = time_aware_split(
        model_df, feature_columns=columns, test_ratio=test_ratio
    )

    metrics_rows = [naive_baseline_metrics(model_df, test_ratio=test_ratio)]
    fitted_models: Dict[str, Pipeline] = {}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for model_name, estimator in get_candidate_models().items():
            pipeline = make_pipeline(estimator, columns)
            pipeline.fit(x_train, y_train)
            predictions = pipeline.predict(x_test)
            row = compute_metrics(y_test, predictions)
            row["Model"] = model_name
            metrics_rows.append(row)
            fitted_models[model_name] = pipeline

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df = metrics_df[["Model", "MAE", "RMSE", "R2"]].sort_values(
        by=["RMSE", "MAE"], ascending=[True, True]
    )
    metrics_df = metrics_df.reset_index(drop=True)

    split_payload = {
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "train_df": train_df,
        "test_df": test_df,
    }
    return metrics_df, fitted_models, split_payload


def tune_regularized_models(
    model_df: pd.DataFrame,
    feature_columns: Optional[Iterable[str]] = None,
    cv_splits: int = 5,
) -> Tuple[pd.DataFrame, Dict[str, GridSearchCV]]:
    """Tune the regularized linear models with time series cross-validation."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    x_train, x_test, y_train, y_test, _, _ = time_aware_split(
        model_df, feature_columns=columns
    )
    cv = TimeSeriesSplit(n_splits=cv_splits)

    search_space = {
        "Lasso Regression": (
            Lasso(max_iter=20000),
            {"model__alpha": [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]},
        ),
        "Ridge Regression": (
            Ridge(),
            {"model__alpha": [0.1, 1.0, 5.0, 10.0, 20.0, 50.0]},
        ),
        "ElasticNet": (
            ElasticNet(max_iter=20000),
            {
                "model__alpha": [0.01, 0.05, 0.1, 0.2, 0.5],
                "model__l1_ratio": [0.3, 0.5, 0.7, 0.9],
            },
        ),
    }

    tuning_rows: List[Dict[str, object]] = []
    tuned_searches: Dict[str, GridSearchCV] = {}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for model_name, (estimator, params) in search_space.items():
            search = GridSearchCV(
                estimator=make_pipeline(estimator, columns),
                param_grid=params,
                scoring="neg_root_mean_squared_error",
                cv=cv,
                n_jobs=1,
            )
            search.fit(x_train, y_train)
            predictions = search.best_estimator_.predict(x_test)
            metrics = compute_metrics(y_test, predictions)
            metrics["Model"] = model_name
            metrics["Best Params"] = str(search.best_params_)
            metrics["CV RMSE"] = float(-search.best_score_)
            tuning_rows.append(metrics)
            tuned_searches[model_name] = search

    tuning_df = pd.DataFrame(tuning_rows)
    tuning_df = tuning_df[
        ["Model", "Best Params", "CV RMSE", "MAE", "RMSE", "R2"]
    ].sort_values(by=["RMSE", "CV RMSE"])
    tuning_df = tuning_df.reset_index(drop=True)
    return tuning_df, tuned_searches


def evaluate_target_conditioning(
    model_df: pd.DataFrame,
    feature_columns: Optional[Iterable[str]] = None,
    alpha: float = 0.2,
) -> pd.DataFrame:
    """Compare raw-target modeling versus log1p target conditioning."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    x_train, x_test, y_train, y_test, _, _ = time_aware_split(
        model_df, feature_columns=columns
    )
    baseline_model = make_pipeline(Lasso(alpha=alpha, max_iter=20000), columns)

    rows: List[Dict[str, float]] = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)

        baseline_model.fit(x_train, y_train)
        raw_predictions = baseline_model.predict(x_test)
        raw_metrics = compute_metrics(y_test, raw_predictions)
        raw_metrics["Target Strategy"] = "Raw Close"
        rows.append(raw_metrics)

        baseline_model.fit(x_train, np.log1p(y_train))
        log_predictions = np.expm1(baseline_model.predict(x_test))
        log_metrics = compute_metrics(y_test, log_predictions)
        log_metrics["Target Strategy"] = "log1p(Close)"
        rows.append(log_metrics)

    result_df = pd.DataFrame(rows)
    result_df = result_df[["Target Strategy", "MAE", "RMSE", "R2"]].sort_values(
        by=["RMSE", "MAE"]
    )
    return result_df.reset_index(drop=True)


def fit_final_model(
    model_df: pd.DataFrame,
    feature_columns: Optional[Iterable[str]] = None,
    preferred_model: str = "Lasso Regression",
) -> Tuple[Pipeline, Dict[str, object], Dict[str, object]]:
    """Tune the preferred model, then fit it on the full modeling frame."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    tuning_df, tuned_searches = tune_regularized_models(model_df, columns)
    best_search = tuned_searches.get(preferred_model)

    if best_search is None:
        raise ValueError("Preferred model '{0}' was not tuned.".format(preferred_model))

    x_train, x_test, y_train, y_test, train_df, test_df = time_aware_split(
        model_df, feature_columns=columns
    )
    holdout_predictions = best_search.best_estimator_.predict(x_test)
    holdout_metrics = compute_metrics(y_test, holdout_predictions)

    final_pipeline = best_search.best_estimator_
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        final_pipeline.fit(model_df[columns], model_df[TARGET_COLUMN])

    payload = {
        "tuning_table": tuning_df,
        "best_params": best_search.best_params_,
        "holdout_metrics": holdout_metrics,
        "train_df": train_df,
        "test_df": test_df,
        "x_test": x_test,
        "y_test": y_test,
        "holdout_predictions": holdout_predictions,
    }
    return final_pipeline, payload, tuned_searches


def model_coefficients(
    fitted_pipeline: Pipeline,
    feature_columns: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Extract absolute coefficient magnitude for linear models."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    estimator = fitted_pipeline.named_steps["model"]
    if not hasattr(estimator, "coef_"):
        raise ValueError("The fitted estimator does not expose coefficients.")

    coef_df = pd.DataFrame(
        {
            "feature": columns,
            "coefficient": estimator.coef_,
        }
    )
    coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
    coef_df = coef_df.sort_values(by="abs_coefficient", ascending=False).reset_index(drop=True)
    return coef_df


def permutation_feature_importance(
    fitted_pipeline: Pipeline,
    x_test: pd.DataFrame,
    y_test: pd.Series,
    feature_columns: Optional[Iterable[str]] = None,
    n_repeats: int = 20,
) -> pd.DataFrame:
    """Compute model-agnostic feature importance on the holdout set."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        result = permutation_importance(
            fitted_pipeline,
            x_test,
            y_test,
            n_repeats=n_repeats,
            random_state=SEED,
            scoring="neg_root_mean_squared_error",
        )

    importance_df = pd.DataFrame(
        {
            "feature": columns,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        }
    )
    importance_df = importance_df.sort_values(
        by="importance_mean", ascending=False
    ).reset_index(drop=True)
    return importance_df


def high_correlation_pairs(
    model_df: pd.DataFrame,
    feature_columns: Optional[Iterable[str]] = None,
    threshold: float = 0.95,
) -> pd.DataFrame:
    """List strongly correlated feature pairs for multicollinearity inspection."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    corr_matrix = model_df[columns].corr().abs()
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    rows: List[Dict[str, object]] = []
    for column in upper_triangle.columns:
        correlations = upper_triangle[column].dropna()
        for index_name, value in correlations.items():
            if value >= threshold:
                rows.append(
                    {
                        "feature_1": index_name,
                        "feature_2": column,
                        "correlation": float(value),
                    }
                )

    return pd.DataFrame(rows).sort_values(by="correlation", ascending=False).reset_index(
        drop=True
    )


def next_forecast_date(frame: pd.DataFrame) -> pd.Timestamp:
    """Return the next monthly period after the latest available record."""
    last_date = pd.to_datetime(frame["Date"]).max()
    return pd.Timestamp(last_date) + pd.offsets.MonthBegin(1)


def build_prediction_frame(
    history_df: pd.DataFrame,
    forecast_date: Union[str, pd.Timestamp],
    open_price: float,
    high_price: float,
    low_price: float,
    feature_columns: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Create a one-row feature frame for interactive scenario prediction."""
    columns = list(feature_columns or FEATURE_COLUMNS)
    if high_price < open_price or low_price > open_price or low_price > high_price:
        raise ValueError(
            "Expected Low <= Open <= High and Low <= High for a valid monthly scenario."
        )

    forecast_timestamp = pd.Timestamp(forecast_date).replace(day=1)
    close_history = history_df["Close"].to_numpy()

    if len(close_history) < 6:
        raise ValueError("At least 6 historical close values are required for prediction.")

    month = forecast_timestamp.month
    feature_values = {
        "Open": float(open_price),
        "High": float(high_price),
        "Low": float(low_price),
        "range_value": float(high_price - low_price),
        "range_pct": float((high_price - low_price) / open_price if open_price else 0.0),
        "high_open_gap": float(high_price - open_price),
        "open_low_gap": float(open_price - low_price),
        "close_lag_1": float(close_history[-1]),
        "close_lag_3": float(close_history[-3]),
        "close_lag_6": float(close_history[-6]),
        "close_ma_3": float(close_history[-3:].mean()),
        "close_ma_6": float(close_history[-6:].mean()),
        "close_std_3": float(pd.Series(close_history[-3:]).std()),
        "close_std_6": float(pd.Series(close_history[-6:]).std()),
        "month_sin": float(np.sin(2 * np.pi * month / 12.0)),
        "month_cos": float(np.cos(2 * np.pi * month / 12.0)),
        "year": int(forecast_timestamp.year),
        "time_index": int(len(history_df)),
        "post_2018_crisis": int(forecast_timestamp >= pd.Timestamp("2018-01-01")),
        "covid_shock": int(forecast_timestamp >= pd.Timestamp("2020-03-01")),
    }

    return pd.DataFrame([feature_values], columns=columns)


def predict_close_price(
    fitted_pipeline: Pipeline,
    history_df: pd.DataFrame,
    forecast_date: Union[str, pd.Timestamp],
    open_price: float,
    high_price: float,
    low_price: float,
    feature_columns: Optional[Iterable[str]] = None,
) -> float:
    """Predict the closing price for a user-supplied monthly scenario."""
    prediction_frame = build_prediction_frame(
        history_df=history_df,
        forecast_date=forecast_date,
        open_price=open_price,
        high_price=high_price,
        low_price=low_price,
        feature_columns=feature_columns,
    )
    prediction = fitted_pipeline.predict(prediction_frame)[0]
    return float(prediction)


### Dataset Loading


In [ ]:
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "yes_bank_regression_azure",
    Path.cwd().parent / "yes_bank_regression_azure",
]

PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / "data" / "yes_bank_stock_prices.csv").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Make sure the notebook is opened with the project folder available."
    )

DATA_PATH = PROJECT_ROOT / "data" / "yes_bank_stock_prices.csv"
raw_df = load_yes_bank_data(DATA_PATH)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)


### Dataset First View


In [ ]:
raw_df.head()


### Dataset Rows & Columns Count


In [ ]:
print(f"Rows    : {raw_df.shape[0]}")
print(f"Columns : {raw_df.shape[1]}")


### Dataset Information


In [ ]:
info_buffer = io.StringIO()
raw_df.info(buf=info_buffer)
print(info_buffer.getvalue())


#### Duplicate Values


In [ ]:
duplicate_count = raw_df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")


#### Missing Values / Null Values


In [ ]:
missing_df = raw_df.isna().sum().to_frame("missing_values")
missing_df["missing_pct"] = (missing_df["missing_values"] / len(raw_df)) * 100
missing_df


### Statistical Summary


In [ ]:
raw_df.describe(include="all").transpose()


### What did we learn from the raw dataset?

- The dataset contains monthly stock records from **July 2005 to November 2020**.
- There are no missing values and no duplicate rows.
- `Open`, `High`, `Low`, and `Close` are numeric and highly related, which is expected for OHLC market data.
- Because this is a short monthly time series, evaluation must preserve chronology instead of using a random split.


## ***2. Understanding Your Variables***


### Variables Description


In [ ]:
variable_description = pd.DataFrame(
    {
        "Variable": ["Date", "Open", "High", "Low", "Close"],
        "Description": [
            "Month of the observation",
            "Opening stock price for the month",
            "Highest stock price touched during the month",
            "Lowest stock price touched during the month",
            "Closing stock price for the month (target variable)",
        ],
    }
)
variable_description


### Target Variable Overview


In [ ]:
target_summary = pd.DataFrame(
    {
        "metric": ["min", "max", "mean", "median", "skewness"],
        "value": [
            raw_df["Close"].min(),
            raw_df["Close"].max(),
            raw_df["Close"].mean(),
            raw_df["Close"].median(),
            raw_df["Close"].skew(),
        ],
    }
)
target_summary


The closing price is positively skewed because the stock experienced a dramatic rise before the later correction phases. We will test target conditioning later, but we should keep the business interpretation on the original rupee scale whenever possible.


## ***3. Dataset Cleanup and Wrangling***


### Data Wrangling Code


In [ ]:
clean_df = raw_df.copy()
clean_df = clean_df.sort_values("Date").reset_index(drop=True)
clean_df["YearMonth"] = clean_df["Date"].dt.to_period("M").astype(str)

print("Is the dataset chronologically sorted?", clean_df["Date"].is_monotonic_increasing)
clean_df.head()


### What cleanup did we perform?

- Converted the `Date` column to proper datetime format.
- Sorted the records chronologically.
- Created a readable monthly label for later charting.
- Since there were no missing values or duplicates, the cleanup stage was intentionally light.


## ***4. EDA, Storytelling, and Chart Experiments***

The goal of EDA here is not just plotting charts, but understanding trend shifts, volatility bursts, seasonality, and the effect of structurally different market periods.


#### Chart 1 - Monthly Closing Price Trend


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(clean_df["Date"], clean_df["Close"], color="#123b5d", linewidth=2.2, label="Close")
plt.axvline(pd.Timestamp("2018-01-01"), linestyle="--", linewidth=1.4, color="#a43a2a", label="2018 stress period")
plt.title("Yes Bank Monthly Closing Price Trend")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.show()


**Insight:** The stock saw a long growth phase followed by a sharp correction, especially after the 2018 stress period. This makes it important to include event-aware features instead of treating the full series as perfectly stable.


#### Chart 2 - Open, High, Low, and Close Comparison


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(clean_df["Date"], clean_df["Open"], label="Open", linewidth=1.8, color="#305f72")
plt.plot(clean_df["Date"], clean_df["High"], label="High", linewidth=1.6, color="#e09f3e")
plt.plot(clean_df["Date"], clean_df["Low"], label="Low", linewidth=1.6, color="#8c2f39")
plt.plot(clean_df["Date"], clean_df["Close"], label="Close", linewidth=2.0, color="#1d3557")
plt.title("OHLC Behavior Over Time")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend(ncol=4)
plt.show()


**Insight:** The OHLC series move together very closely, which is useful for prediction but also creates strong multicollinearity. We will inspect that explicitly before modeling.


#### Chart 3 - Seasonality View by Month


In [ ]:
seasonality_df = clean_df.copy()
seasonality_df["Month"] = seasonality_df["Date"].dt.strftime("%b")
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

plt.figure(figsize=(13, 5))
sns.boxplot(data=seasonality_df, x="Month", y="Close", order=month_order, color="#5c88b4")
plt.title("Distribution of Closing Price by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Closing Price")
plt.show()


**Insight:** There is no clean seasonal pattern strong enough to dominate the series, but month encoding can still help the model capture softer cyclical effects.


#### Chart 4 - Moving Averages and Volatility Proxy


In [ ]:
eda_df = clean_df.copy()
eda_df["close_ma_3"] = eda_df["Close"].rolling(3).mean()
eda_df["close_ma_12"] = eda_df["Close"].rolling(12).mean()
eda_df["range_value"] = eda_df["High"] - eda_df["Low"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(eda_df["Date"], eda_df["Close"], label="Close", color="#143d59", linewidth=2)
axes[0].plot(eda_df["Date"], eda_df["close_ma_3"], label="3M Moving Average", color="#f4b942", linewidth=1.8)
axes[0].plot(eda_df["Date"], eda_df["close_ma_12"], label="12M Moving Average", color="#6c7a89", linewidth=1.8)
axes[0].set_title("Trend Smoothing with Moving Averages")
axes[0].legend()

axes[1].bar(eda_df["Date"], eda_df["range_value"], color="#9d4d4d", alpha=0.8)
axes[1].set_title("Monthly Trading Range (High - Low)")
axes[1].set_ylabel("Range")

plt.tight_layout()
plt.show()


**Insight:** Moving averages smooth the long-term trend, while the monthly range highlights the stress periods where volatility expanded materially.


#### Chart 5 - Pre-2018 vs Post-2018 Closing Price Distribution


In [ ]:
comparison_df = clean_df.copy()
comparison_df["Regime"] = np.where(comparison_df["Date"] < pd.Timestamp("2018-01-01"), "Before 2018", "2018 and After")

plt.figure(figsize=(8, 5))
sns.boxplot(data=comparison_df, x="Regime", y="Close", palette=["#7a9e9f", "#bc4b51"])
plt.title("Closing Price Distribution Across Market Regimes")
plt.xlabel("")
plt.ylabel("Closing Price")
plt.show()


**Insight:** The distribution changes visibly after 2018, which supports the decision to add structural-break indicator variables in the feature engineering stage.


#### Chart 6 - Correlation Heatmap


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(clean_df[["Open", "High", "Low", "Close"]].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation Heatmap of Raw Price Variables")
plt.show()


**Insight:** All core price variables are strongly correlated, which is useful for predictive accuracy but needs thoughtful handling to keep the final model stable and interpretable.


## ***5. Feature Engineering and Pre-processing***


### Feature Creation, Encoding, and Modeling Frame


In [ ]:
model_df = build_modeling_frame(clean_df)
print("Modeling rows available after lag/rolling feature creation:", len(model_df))
print("Total engineered features used:", len(FEATURE_COLUMNS))
model_df.head()


**Feature engineering choices made in this project**

- Time features: `year`, `time_index`
- Cyclical encoding: `month_sin`, `month_cos`
- Price spread features: `range_value`, `range_pct`, `high_open_gap`, `open_low_gap`
- Lag features: prior monthly closes
- Rolling features: 3-month and 6-month moving averages and standard deviations
- Structural break indicators: `post_2018_crisis`, `covid_shock`


### Multicollinearity Inspection


In [ ]:
correlation_pairs = high_correlation_pairs(model_df, FEATURE_COLUMNS, threshold=0.95)
correlation_pairs.head(15)


**Multicollinearity handling strategy**

- OHLC and lag features are naturally correlated in financial data.
- Instead of dropping every strong predictor, we keep business-relevant variables and control coefficient instability through **L1/L2 regularization**.
- This gives a better interview narrative than blindly deleting most of the signal.


### Target Feature Conditioning


In [ ]:
conditioning_df = evaluate_target_conditioning(model_df, FEATURE_COLUMNS, alpha=0.2)
conditioning_df


**Decision:** The raw target performs better than `log1p(Close)` on the holdout set, so the final model keeps the target in the original scale for easier business interpretation.


### Train-Test Split and Scaling Strategy


In [ ]:
x_train, x_test, y_train, y_test, train_df, test_df = time_aware_split(model_df, FEATURE_COLUMNS, test_ratio=0.2)

split_summary = pd.DataFrame(
    {
        "split": ["Train", "Test"],
        "start_date": [train_df["Date"].min(), test_df["Date"].min()],
        "end_date": [train_df["Date"].max(), test_df["Date"].max()],
        "rows": [len(train_df), len(test_df)],
    }
)
split_summary


The split is chronological, not random. Feature scaling is applied inside the model pipeline, which avoids data leakage from the test period.


## ***6. Model Implementation***


### Baseline and Candidate Model Comparison


In [ ]:
metrics_df, fitted_models, split_payload = evaluate_models(model_df, FEATURE_COLUMNS, test_ratio=0.2)
metrics_df


**Modeling takeaway:** Regularized linear models outperform the tree-based models here. That means the relationship between the closing price and the engineered monthly price features is mostly linear and very strong.


### Time-Series-Only vs Same-Month Estimation Framing


In [ ]:
lag_only_features = [
    "close_lag_1",
    "close_lag_3",
    "close_lag_6",
    "close_ma_3",
    "close_ma_6",
    "close_std_3",
    "close_std_6",
    "month_sin",
    "month_cos",
    "year",
    "time_index",
    "post_2018_crisis",
    "covid_shock",
]

lag_only_metrics, _, _ = evaluate_models(model_df, lag_only_features, test_ratio=0.2)

framing_comparison = pd.DataFrame(
    {
        "Framing": ["Final feature set", "Lag-only feature set"],
        "Lasso RMSE": [
            metrics_df.loc[metrics_df["Model"] == "Lasso Regression", "RMSE"].iloc[0],
            lag_only_metrics.loc[lag_only_metrics["Model"] == "Lasso Regression", "RMSE"].iloc[0],
        ],
        "Lasso R2": [
            metrics_df.loc[metrics_df["Model"] == "Lasso Regression", "R2"].iloc[0],
            lag_only_metrics.loc[lag_only_metrics["Model"] == "Lasso Regression", "R2"].iloc[0],
        ],
    }
)
framing_comparison


**Interpretation:** A pure lag-only setup performs noticeably worse. That is why the final project should be presented as a **monthly close estimation system** using the month's trading range, not as a pure ahead-of-time trading forecast.


### Hyperparameter Tuning and Regularization


In [ ]:
tuning_df, tuned_searches = tune_regularized_models(model_df, FEATURE_COLUMNS)
tuning_df


**Why regularization matters here**

- The data contains many correlated predictors.
- Lasso helps shrink weaker coefficients and improves interpretability.
- Ridge and ElasticNet act as stability-oriented alternatives.


### Final Model Fit


In [ ]:
final_model, final_payload, tuned_searches = fit_final_model(model_df, FEATURE_COLUMNS)

final_metrics_df = pd.DataFrame([final_payload["holdout_metrics"]])
final_metrics_df


### Actual vs Predicted Performance on the Holdout Set


In [ ]:
holdout_results = pd.DataFrame(
    {
        "Date": final_payload["test_df"]["Date"].to_numpy(),
        "Actual": final_payload["y_test"].to_numpy(),
        "Predicted": final_payload["holdout_predictions"],
    }
)
holdout_results["Residual"] = holdout_results["Actual"] - holdout_results["Predicted"]
holdout_results.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(holdout_results["Date"], holdout_results["Actual"], label="Actual", linewidth=2, color="#123b5d")
axes[0].plot(holdout_results["Date"], holdout_results["Predicted"], label="Predicted", linewidth=2, color="#d3872b")
axes[0].set_title("Actual vs Predicted Close Price")
axes[0].legend()

axes[1].scatter(holdout_results["Predicted"], holdout_results["Residual"], color="#8f3b45", alpha=0.8)
axes[1].axhline(0, linestyle="--", color="black", linewidth=1)
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Predicted Close")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()


## ***7. Model Explainability***


### Coefficient-Based Importance and Permutation Importance


In [ ]:
coefficient_df = model_coefficients(final_model, FEATURE_COLUMNS)
importance_df = permutation_feature_importance(
    final_model,
    final_payload["x_test"],
    final_payload["y_test"],
    FEATURE_COLUMNS,
)

display(coefficient_df.head(10))
display(importance_df.head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

coef_plot = coefficient_df.head(8).iloc[::-1]
axes[0].barh(coef_plot["feature"], coef_plot["abs_coefficient"], color="#19486a")
axes[0].set_title("Top Absolute Coefficients")
axes[0].set_xlabel("Absolute Coefficient")

perm_plot = importance_df.head(8).iloc[::-1]
axes[1].barh(perm_plot["feature"], perm_plot["importance_mean"], color="#d3872b")
axes[1].set_title("Top Permutation Importances")
axes[1].set_xlabel("Importance Mean")

plt.tight_layout()
plt.show()


**Explainability summary**

- Current-month price levels and spread-related features dominate prediction quality.
- Structural break indicators help the model account for post-2018 and post-COVID behavior shifts.
- The final Lasso model is explainable enough for interview discussion while still giving strong holdout performance.


## ***8. Deployment - Streamlit + Microsoft Azure GenAI***


### Deployment Strategy

- Build an interactive Streamlit app for historical trend analysis, model metrics, and scenario-based prediction.
- Integrate **Azure OpenAI** so users can ask natural-language questions such as:
  - Why was Lasso chosen?
  - What changed after 2018?
  - Is this a forecast or an estimation model?
- Keep Azure credentials in environment variables rather than hardcoding them.


In [ ]:
print("Streamlit launch command:")
print("streamlit run app.py")

print("\nRecommended Azure environment variables:")
print("AZURE_OPENAI_ENDPOINT=https://YOUR-RESOURCE-NAME.openai.azure.com")
print("AZURE_OPENAI_API_KEY=your_api_key_here")
print("AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4.1-mini")


## ***9. Conclusion***


- The dataset is clean, compact, and highly informative despite having only monthly observations.
- Yes Bank shows a visible regime shift after 2018, so event-aware features improve the storytelling and modeling logic.
- Regularized linear models outperform tree-based models for this problem.
- A lag-only, pure time-series framing is weaker than the final feature set, so the best business framing is **monthly close estimation** using the month's trading range.
- The project is deployment-ready through Streamlit, and Azure GenAI adds a strong presentation layer for interviews.


## ***10. Suggested Git Commit Checkpoints***


1. `git commit -m "feat: add yes bank stock dataset and project scaffold"`
2. `git commit -m "feat: complete yes bank eda and feature engineering notebook sections"`
3. `git commit -m "feat: add regression modeling, tuning, and explainability workflow"`
4. `git commit -m "feat: build streamlit app with azure genai integration"`
5. `git commit -m "docs: add project readme and deployment notes"`


### ***Hurrah! You have successfully completed the project notebook structure.***
